In [74]:
import math

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

# Utils

In [75]:
def train_test_split(data, ratio=0.8):
    train_size = int(math.floor(len(data) * ratio))
    train_data = data[:train_size]
    test_data = data[train_size:]
    return train_data, test_data


In [76]:
def norm_rows(df_input):
    df_scaled = df_input.copy()

    def scale_row(row):
        row_min = row.min()
        row_max = row.max()
        row_range = row_max - row_min

        if pd.isna(row_min):
            return row

        if row_range == 0:
            return row.apply(lambda x: 0.0 if pd.notna(x) else np.nan)
        else:
            return (row - row_min) / row_range

    df_scaled = df_scaled.apply(scale_row, axis=1)

    return df_scaled

In [77]:
df = pd.read_csv("./data/customer_led_network_revolution/cond_data/Leicester.csv")
df.drop(["dt", "timezone", "city_name", "lat", "lon", "visibility", "sea_level", "grnd_level", "wind_gust", "rain_1h", "rain_3h", "snow_1h", "snow_3h", "weather_description", "weather_icon", "weather_id"], axis=1, inplace=True)
df = pd.get_dummies(df, columns=['weather_main'], prefix='weather', dtype=int)
dt_iso_cleaned = df['dt_iso'].str.replace(' UTC', '', regex=False)
df['date'] = pd.to_datetime(dt_iso_cleaned, format='%Y-%m-%d %H:%M:%S %z')
df['date'] = df['date'].dt.strftime('%d/%m/%Y %H:%M:%S')
df.drop(['dt_iso'], axis=1, inplace=True)

print(df["date"].min(), df["date"].max())

print(df.shape)
df.isna().sum()

01/01/1979 00:00:00 31/12/2023 23:00:00
(401880, 15)


temp              0
dew_point         0
feels_like        0
temp_min          0
temp_max          0
pressure          0
humidity          0
wind_speed        0
wind_deg          0
clouds_all        0
weather_Clear     0
weather_Clouds    0
weather_Rain      0
weather_Snow      0
date              0
dtype: int64

In [78]:
df.head()

,temp,dew_point,feels_like,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,clouds_all,weather_Clear,weather_Clouds,weather_Rain,weather_Snow,date
0,267.40,263.94,260.83,266.51,267.66,1012,74,5.32,348,10,1,0,0,0,01/01/1979 00:00:00
1,267.32,263.56,260.74,266.29,267.55,1013,72,5.30,348,5,1,0,0,0,01/01/1979 01:00:00
2,267.24,263.32,260.69,267.03,267.45,1014,71,5.22,344,0,1,0,0,0,01/01/1979 02:00:00
3,266.14,263.02,259.33,266.01,266.28,1015,76,5.20,342,1,1,0,0,0,01/01/1979 03:00:00
4,266.07,262.95,259.20,265.89,266.51,1016,76,5.26,340,1,1,0,0,0,01/01/1979 04:00:00


In [79]:
df["date"].min(), df["date"].max()

('01/01/1979 00:00:00', '31/12/2023 23:00:00')

In [82]:
start_date_str = '01/01/2012 00:00:00'
end_date_str = '31/12/2012 23:30:00'

start_datetime = pd.to_datetime(start_date_str)
end_datetime = pd.to_datetime(end_date_str)

df["date"] = pd.to_datetime(df["date"], format='%d/%m/%Y %H:%M:%S')

date_filter_mask = (df['date'] >= start_datetime) & \
                   (df['date'] <= end_datetime)
                   
df = df[date_filter_mask]

C:\Users\Arne\AppData\Local\Temp\ipykernel_19460\3096985692.py:5: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  end_datetime = pd.to_datetime(end_date_str)


In [83]:
df["date"].min(), df["date"].max()

(Timestamp('2012-01-01 00:00:00'), Timestamp('2012-12-31 23:00:00'))

In [84]:
df.set_index("date", inplace=True)

In [85]:
df = df.T

In [86]:
df

date,2012-01-01 00:00:00,2012-01-01 01:00:00,2012-01-01 02:00:00,2012-01-01 03:00:00,2012-01-01 04:00:00,2012-01-01 05:00:00,2012-01-01 06:00:00,2012-01-01 07:00:00,2012-01-01 08:00:00,2012-01-01 09:00:00,...,2012-12-31 14:00:00,2012-12-31 15:00:00,2012-12-31 16:00:00,2012-12-31 17:00:00,2012-12-31 18:00:00,2012-12-31 19:00:00,2012-12-31 20:00:00,2012-12-31 21:00:00,2012-12-31 22:00:00,2012-12-31 23:00:00
temp,285.15,284.35,284.20,283.44,284.05,283.32,283.19,283.12,282.44,282.49,...,283.29,283.35,282.97,282.78,282.39,281.90,281.34,280.33,280.32,280.34
dew_point,283.73,282.94,282.62,281.54,281.80,281.09,280.96,280.89,280.39,280.61,...,282.84,283.05,282.67,282.48,281.16,281.60,280.28,279.43,279.27,279.13
feels_like,284.78,283.90,283.71,282.82,283.44,282.64,282.50,280.10,279.21,279.26,...,282.89,282.98,281.40,281.17,279.46,280.11,278.64,277.64,277.94,277.93
temp_min,284.18,283.28,282.98,283.36,282.98,282.88,282.39,281.38,281.49,282.36,...,282.78,282.78,282.58,282.28,281.88,281.58,280.71,280.31,279.78,280.01
temp_max,285.36,284.99,284.69,283.99,284.36,283.51,283.36,283.36,282.91,282.84,...,283.49,284.29,283.36,283.36,283.29,282.49,282.49,282.27,281.27,280.68
pressure,1004.00,1005.00,1004.00,1004.00,1004.00,1004.00,1004.00,1004.00,1004.00,1003.00,...,995.00,994.00,995.00,995.00,995.00,996.00,996.00,997.00,997.00,998.00
humidity,91.00,91.00,90.00,88.00,86.00,86.00,86.00,86.00,87.00,88.00,...,97.00,98.00,98.00,98.00,92.00,98.00,93.00,94.00,93.00,92.00
wind_speed,6.99,7.57,7.44,7.23,6.99,6.91,6.71,6.82,6.90,6.95,...,4.63,3.60,3.08,3.08,5.91,3.08,4.62,4.09,3.52,3.57
wind_deg,228.00,232.00,233.00,232.00,230.00,229.00,225.00,222.00,219.00,214.00,...,219.00,219.00,218.00,212.00,238.00,221.00,248.00,252.00,260.00,264.00
clouds_all,100.00,100.00,100.00,100.00,94.00,90.00,97.00,84.00,72.00,75.00,...,100.00,80.00,100.00,100.00,99.00,92.00,52.00,30.00,54.00,65.00


In [90]:
norm_df.to_csv("./data/customer_led_network_revolution/preprocessed/cond_df.csv")
